# Phase 2 verification - `myroom`

Confirms the strip-to-core **Phase 2** changes did not regress static quality:
- `fourier_K=0` in static mode (freed GPU memory; params had LR=0 anyway)
- static export now writes **one** clean frame instead of running the placeholder deform per timestamp

Closes task **P2-V**. Runtime: **A100 GPU** (Runtime -> Change runtime type). ~10-25 min.


## 0. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 1. Clone repo (branch `chore/strip-to-core`)

In [ ]:
import os
REPO_URL = "https://github.com/mehmettahacumurcu/gaussian-splatter.git"
BRANCH   = "chore/strip-to-core"   # Phase 2 work lives here, NOT main
if not os.path.exists("gaussian-splatter"):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL}
%cd gaussian-splatter
!git log --oneline -1

## 2. Environment (deps + gsplat JIT build)

In [ ]:
!bash colab/bootstrap.sh --colmap

In [ ]:
# If this errors about numpy: Runtime -> Restart session, then re-run THIS cell only.
import torch, gsplat
print("torch", torch.__version__, "| gsplat", gsplat.__version__,
      "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 3. Data - mount Drive and link `myroom`

Upload the **entire** pre-processed `myroom` folder to your Drive once — not just a couple of
subfolders. The pipeline needs `video.mp4` (or an `images/` photo set) as its input, and it reuses
the cached COLMAP + depth via the hidden `.cache_markers/` dir, so upload **all** of it:

```
myroom/
  video.mp4          <- the input (this scene has NO images/ folder)
  frames/            <- 304 extracted frames (cache hit)
  colmap/            <- poses + sparse (cache hit)
  .cache_markers/    <- REQUIRED: lets COLMAP + depth cache-hit instead of recomputing
  depth/             <- Metric3D depth, used by --foundation (cache hit)
```

Why the whole folder: `_resolve_input` only accepts `images/` or `video.mp4` (a bare `frames/`
dir is rejected), and COLMAP reuse needs `.cache_markers/colmap.json`. With those present and the
`balanced` preset, COLMAP and depth are cache hits (no recompute). Edit `DRIVE_MYROOM` below to match.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MYROOM = "/content/drive/MyDrive/4dgs/myroom"   # <-- EDIT to your Drive path
import os
os.makedirs("data", exist_ok=True)
target = "data/myroom"
if not (os.path.islink(target) or os.path.exists(target)):
    os.symlink(DRIVE_MYROOM, target)
print("linked:", os.path.realpath(target))
!ls -la data/myroom

## 4. Run static 3DGS + NVS eval

`balanced` preset is plenty for a no-regression check. We pass `--foundation` so the run uses the
**cached** depth supervision — this matches the original ~29 dB depth-supervised baseline, so the
27 dB gate floor is apples-to-apples. (An RGB-only run, i.e. without `--foundation`, can legitimately
land a few dB lower and trip the floor with a misleading FAIL.) Depth is a cache hit via
`.cache_markers/`, so `--foundation` adds no recompute here.

In [ ]:
!python scripts/static_3dgs.py --scene myroom --preset balanced --foundation --nvs-eval

## 5. Verify (no-regression gate)

In [ ]:
import sys
sys.path.insert(0, ".")
from colab.verify_helpers import check_phase2, gpu_mem_summary
gpu_mem_summary()
ok = check_phase2("myroom", baseline_psnr=29.0, min_psnr=27.0)
print("\nP2-V:", "PASS - Phase 2 confirmed" if ok else "FAIL - investigate above")

## 6. Save to Drive - splat (always) + results + walkable world

The splat `.ply` is copied first and unconditionally, so you have it even if the world-wrap step errors.

In [ ]:
import sys
sys.path.insert(0, ".")
from colab.verify_helpers import save_splat_to_drive, copy_results_to_drive, wrap_and_save_world

# (a) raw splat .ply - saved FIRST, always (open in any 3DGS viewer)
save_splat_to_drive("myroom", "/content/drive/MyDrive/4dgs")
# (b) full results (eval json, logs, orbit.mp4)
copy_results_to_drive("myroom", "/content/drive/MyDrive/4dgs/results")
# (c) walkable world bundle - best effort; the splat above is safe even if this fails
try:
    wrap_and_save_world("myroom", "/content/drive/MyDrive/4dgs")
except Exception as e:
    print("  world wrap skipped:", e, "\n  -> the splat .ply above is still on Drive")

**View it.**
- *Splat only* (simplest): open `MyDrive/4dgs/splats/myroom.ply` in any 3DGS viewer, e.g.
  https://superspl.at/editor - no project needed.
- *Walkable*: download `MyDrive/4dgs/worlds/myroom/` into your local `worlds/myroom/`,
  run `cd frontend && npm run dev`, open the **Interactive** page, pick **"My Room (static 3DGS, Colab)"** in the
  world dropdown (top-right).